In [1]:
import os
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_postgres.vectorstores import PGVector

load_dotenv()

embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small" # 1536차원 작은 모델
)

connection_url = os.getenv("DATABASE_URL")

vector_store = PGVector(
    embeddings = embedding_model,
    connection=connection_url,
    collection_name="library_rules",
    use_jsonb= True # 메타데이터를 JSONB로 저장
)

In [2]:
# 지난 실습에서 분할했던 도서관 규정 청크 리스트를 정의합니다.
library_rule_chunks = [
    "[2026학년도 중앙도서관 대출 및 열람 규정]\n제1조(대출 자격) 본교 재학생 및 휴학생은 학생증을 제시한 후 도서를 대출할 수 있습니다.",
    "제2조(대출 권수 및 기간) 학부생은 1인당 최대 5권까지 대출할 수 있으며 대출 기간은 14일입니다.\n단, 학술 연구용 특수 도서는 7일간 대출 가능하며 연장은 불가능합니다.",
    "제3조(연체 및 제재) 반납 기한을 초과할 경우 연체 일수만큼 대출이 정지됩니다."
]

saved_ids = vector_store.add_texts(
    texts=library_rule_chunks
)

chunk_ids = ["rule_chunk_001", "rule_chunk_002", "rule_chunk_003"]

print(f"저장 갯수: {len(saved_ids)}")
print(f"청크 ID: {saved_ids}")



저장 갯수: 3
청크 ID: ['8e7b3102-660b-4a79-82d4-72d7f447c8cd', 'bd163413-bf7c-4573-a9e4-e09a6d4d97f8', '1b17b97e-829f-449b-a4f3-df12d65227f4']


In [6]:
question = "특수 도서는 며칠 동안 대출 할 수 있나요?"

search_results = vector_store.similarity_search(
    query= question, k=1 #top-k 설정
)

print(search_results[0].page_content)

제2조(대출 권수 및 기간) 학부생은 1인당 최대 5권까지 대출할 수 있으며 대출 기간은 14일입니다.
단, 학술 연구용 특수 도서는 7일간 대출 가능하며 연장은 불가능합니다.
